# Diagnose, Debugging und Health Checks

Diagnose beginnt mit Status, Events und Logs. Danach werden Konfiguration, Netzwerk und Ressourcen untersucht. Monitoring (Kapitel 12) liefert längerfristige Metriken; kubectl dient der unmittelbaren Untersuchung.

## Voraussetzungen

Python-Kernel, kubectl und K3s-Trainingscluster. Aus diesem Kapitelverzeichnis ausführen. Die verwendeten Demo-Ressourcen müssen frei sein. Aktiven Kontext und Namespace prüfen.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Typische Fehler erzeugen

Die drei Pods demonstrieren ImagePullBackOff, CrashLoopBackOff und OOMKilled. Der Speicherverbrauch ist durch ein kleines Memory-Limit begrenzt. Status kann zunächst ContainerCreating oder ErrImagePull sein.

In [ ]:
%%bash
set -eo pipefail
cat errors.yml
kubectl apply -f errors.yml

In [ ]:
%%bash
set -eo pipefail
kubectl get pods -l app=diag-demo
kubectl describe pod diag-image
kubectl describe pod diag-crash
kubectl describe pod diag-oom

## 2. Logs und Events

Events erklären z.B. Image-Pull-Fehler; Logs zeigen Anwendungsfehler. --previous liest Logs der vorherigen Container-Instanz, sofern vorhanden. Bei noch nicht gestarteten Containern gibt es keine Logs.

In [ ]:
%%bash
set -eo pipefail
kubectl get events --sort-by=.metadata.creationTimestamp
kubectl logs diag-crash || true
kubectl logs diag-crash --previous || true
kubectl get pod diag-oom -o jsonpath='{.status.containerStatuses[0].state.terminated.reason}'
echo

## 3. Health Checks

Startup Probe schützt langsam startende Anwendungen vor vorzeitigen Liveness-Prüfungen. Liveness-Fehler führen nach Erreichen des Schwellwerts zum Container-Neustart. Readiness-Fehler entfernen den Pod aus den regulären Service-Backends, ohne ihn neu zu starten.

In [ ]:
%%bash
set -eo pipefail
cat probes.yml
kubectl apply -f probes.yml
kubectl wait --for=jsonpath='{.status.phase}'=Running pod/diag-probes --timeout=180s
kubectl describe pod diag-probes
kubectl get endpointslice -l kubernetes.io/service-name=diag-probes -o yaml

## 4. Readiness reparieren

Die Datei ready.html fehlt absichtlich. Mit exec erzeugen; anschließend sollte Ready=True sein. Die EndpointSlice enthält dann einen bereiten Endpoint.

In [ ]:
%%bash
set -eo pipefail
kubectl exec diag-probes -- sh -c 'echo ready > /usr/share/nginx/html/ready.html'
kubectl wait --for=condition=Ready pod/diag-probes --timeout=90s
kubectl get endpointslice -l kubernetes.io/service-name=diag-probes -o yaml

## 5. Readiness erneut verlieren

Datei entfernen. Der Pod läuft weiter, wird aber nicht mehr als bereit betrachtet.

In [ ]:
%%bash
set -eo pipefail
kubectl exec diag-probes -- rm /usr/share/nginx/html/ready.html
kubectl wait --for=condition=Ready=false pod/diag-probes --timeout=90s
kubectl get pod diag-probes
kubectl get endpointslice -l kubernetes.io/service-name=diag-probes -o yaml

## 6. Aufräumen

Alle absichtlich fehlerhaften Ressourcen entfernen.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f probes.yml -f errors.yml
kubectl get pods -l app=diag-demo

## Fazit

Status, Events, Logs und Probes beantworten unterschiedliche Diagnosefragen. Pending und Selector-Probleme wurden bereits in Scheduling und Services demonstriert.